In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# read pcd file into a structured array
pcd_file = Path.cwd() / "../data/pcd_files/00000.pcd"


def parse_pcd_header(file: Path) -> np.dtype:
    fields: list[str] = []
    sizes: list[int] = []
    types: list[str] = []
    # counts: list[int] = []
    with file.open("r") as f:
        for i, line in enumerate(f):
            if line.startswith("FIELDS"):
                fields = [t.strip() for t in line.split(" ")[1:]]
            elif line.startswith("SIZE"):
                sizes = [int(t) for t in line.split(" ")[1:]]
            elif line.startswith("TYPE"):
                types = [t.strip() for t in line.split(" ")[1:]]
            # elif line.startswith("COUNT"):
            #     counts = [int(t) for t in line.split(" ")[1:]]

            if i > 10:
                break

    dtypes = [(n, f"<{t.lower()}{s}") for n, t, s in zip(fields, types, sizes)]
    return np.dtype(dtypes)


def read_pcd_ascii(file: Path) -> np.ndarray:
    return np.loadtxt(file, skiprows=11, dtype=parse_pcd_header(file))


dtype = parse_pcd_header(pcd_file)
print(dtype)

pcd = read_pcd_ascii(pcd_file)

In [ ]:
beam1 = pcd[pcd["beam"] == 1]
beam2 = pcd[pcd["beam"] == 2]

# plot one line for each beam, with x beam["fire"] and y beam["elev"]
fig = make_subplots(rows=1, cols=1)

for i in range(8):
    beam = pcd[pcd["beam"] == i]
    fig.add_trace(
        go.Scatter(x=beam["fire"], y=beam["elev"], mode="lines", name=f"Beam {i}")
    )
fig.show()
